In [ ]:
# ============================================================
# Member 3 - XGBoost + DistilBERT
# Project: Research Paper Subject Category Prediction
# Dataset: arxiv_15000_balanced.csv
# ============================================================


In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

import torch
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

print("Libraries imported successfully")
print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

possible_paths = [
    Path("../data/processed/arxiv_15000_balanced.csv"),
    Path("data/processed/arxiv_15000_balanced.csv"),
    Path("../data/processed/arxiv_107892_balanced.csv"),
    Path("data/processed/arxiv_107892_balanced.csv")
]

dataset_path = None

for path in possible_paths:
    if path.exists():
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset file not found. Check data/processed folder."
    )

print("Dataset path:", dataset_path)

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()


In [ ]:
# ============================================================
# 3. HANDLE CATEGORY COLUMN
# ============================================================

def map_arxiv_category(categories):
    if pd.isna(categories):
        return None

    categories = str(categories).split()

    for cat in categories:
        if cat.startswith("cs."):
            return "Computer Science"
        elif cat.startswith("math."):
            return "Mathematics"
        elif cat.startswith("stat."):
            return "Statistics"
        elif cat.startswith("q-bio."):
            return "Quantitative Biology"
        elif cat.startswith("q-fin."):
            return "Quantitative Finance"
        elif (
            cat.startswith("physics.") or
            cat.startswith("astro-ph") or
            cat.startswith("cond-mat") or
            cat.startswith("gr-qc") or
            cat.startswith("hep-") or
            cat.startswith("nucl-") or
            cat.startswith("quant-ph")
        ):
            return "Physics"

    return None

if "main_category" not in df.columns:
    print("main_category column not found. Creating main_category from categories...")
    if "categories" not in df.columns:
        raise KeyError("Both main_category and categories columns are missing.")
    df["main_category"] = df["categories"].apply(map_arxiv_category)

print("Category column ready")
print(df["main_category"].value_counts())


In [ ]:
# ============================================================
# 4. CLEAN DATASET
# ============================================================

required_columns = ["title", "abstract", "main_category"]

for col in required_columns:
    if col not in df.columns:
        raise KeyError(f"Required column missing: {col}")

df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")
df["main_category"] = df["main_category"].fillna("Unknown")

df = df[df["main_category"] != "Unknown"]
df = df.drop_duplicates(subset=["title", "abstract"])

df["text"] = df["title"] + " " + df["abstract"]

print("After cleaning:", df.shape)
print(df["main_category"].value_counts())


In [ ]:
# ============================================================
# 5. CLASS DISTRIBUTION CHART
# ============================================================

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="main_category")
plt.xticks(rotation=45)
plt.title("Class Distribution of arXiv Dataset")
plt.xlabel("Subject Category")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 6. PREPARE DATA FOR XGBOOST
# ============================================================

X = df["text"]
y = df["main_category"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


In [ ]:
# ============================================================
# 7. TF-IDF FEATURE EXTRACTION
# ============================================================

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)


In [ ]:
# ============================================================
# 8. TRAIN XGBOOST MODEL
# ============================================================

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_tfidf, y_train)

print("XGBoost training completed")


In [ ]:
# ============================================================
# 9. EVALUATE XGBOOST MODEL
# ============================================================

y_pred = xgb_model.predict(X_test_tfidf)

xgb_accuracy = accuracy_score(y_test, y_pred)
xgb_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
xgb_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
xgb_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("XGBoost Evaluation Results")
print("--------------------------")
print("Accuracy :", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall   :", xgb_recall)
print("F1 Score :", xgb_f1)

print("\nXGBoost Classification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))


In [ ]:
# ============================================================
# 10. XGBOOST CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 11. SAVE XGBOOST MODEL
# ============================================================

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(xgb_model, "../models/member3_xgboost_model.pkl")
joblib.dump(tfidf, "../models/member3_tfidf_vectorizer.pkl")
joblib.dump(label_encoder, "../models/member3_label_encoder.pkl")

print("Member 3 XGBoost model saved successfully")


In [ ]:
# ============================================================
# 12. XGBOOST SINGLE PREDICTION FUNCTION
# ============================================================

def predict_category_xgboost(title, abstract):
    text = title + " " + abstract
    text_tfidf = tfidf.transform([text])

    prediction = xgb_model.predict(text_tfidf)[0]
    probabilities = xgb_model.predict_proba(text_tfidf)[0]

    predicted_category = label_encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction] * 100

    return predicted_category, confidence

sample_title = "Deep Learning Based Text Classification"
sample_abstract = """
This paper proposes a neural network model for classifying scientific documents
using transformer based language representations.
"""

category, confidence = predict_category_xgboost(sample_title, sample_abstract)

print("XGBoost Prediction")
print("------------------")
print("Predicted Category:", category)
print("Confidence:", round(confidence, 2), "%")


In [ ]:
# ============================================================
# 13. XGBOOST TOP-3 PREDICTION FUNCTION
# ============================================================

def predict_top3_xgboost(title, abstract):
    text = title + " " + abstract
    text_tfidf = tfidf.transform([text])

    probabilities = xgb_model.predict_proba(text_tfidf)[0]
    top3_indices = np.argsort(probabilities)[-3:][::-1]

    results = []

    for idx in top3_indices:
        category = label_encoder.inverse_transform([idx])[0]
        confidence = probabilities[idx] * 100
        results.append((category, round(confidence, 2)))

    return results

top3 = predict_top3_xgboost(sample_title, sample_abstract)

print("XGBoost Top-3 Predictions")
print("-------------------------")
for category, confidence in top3:
    print(category, ":", confidence, "%")


In [ ]:
# ============================================================
# 14. CREATE SMALL BALANCED DATASET FOR DISTILBERT
# ============================================================

records_per_class = 500

sampled_parts = []

for category in df["main_category"].unique():
    category_df = df[df["main_category"] == category]

    sample_size = min(len(category_df), records_per_class)

    sampled_category_df = category_df.sample(
        n=sample_size,
        random_state=42
    )

    sampled_parts.append(sampled_category_df)

df_small = pd.concat(sampled_parts, ignore_index=True)

print("Small dataset shape:", df_small.shape)
print(df_small["main_category"].value_counts())


In [ ]:
# ============================================================
# 15. PREPARE DATA FOR DISTILBERT
# ============================================================

bert_label_encoder = LabelEncoder()

df_small["label"] = bert_label_encoder.fit_transform(df_small["main_category"])

print("DistilBERT Classes:")
print(bert_label_encoder.classes_)
print("Number of classes:", len(bert_label_encoder.classes_))

train_df, test_df = train_test_split(
    df_small[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df_small["label"]
)

print("DistilBERT train size:", train_df.shape)
print("DistilBERT test size:", test_df.shape)


In [ ]:
# ============================================================
# 16. CONVERT TO HUGGINGFACE DATASET
# ============================================================

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(train_dataset)
print(test_dataset)


In [ ]:
# ============================================================
# 17. TOKENIZATION
# ============================================================

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

print(train_dataset)
print(test_dataset)


In [ ]:
# ============================================================
# 18. LOAD DISTILBERT MODEL
# ============================================================

num_labels = len(bert_label_encoder.classes_)

distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("DistilBERT model loaded successfully")


In [ ]:
# ============================================================
# 19. DISTILBERT METRICS FUNCTION
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall": recall_score(labels, predictions, average="macro", zero_division=0),
        "f1": f1_score(labels, predictions, average="macro", zero_division=0)
    }


In [ ]:
# ============================================================
# 20. TRAINING ARGUMENTS
# ============================================================

try:
    training_args = TrainingArguments(
        output_dir="../models/member3_distilbert_results",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_dir="../models/member3_distilbert_logs",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none"
    )

except TypeError:
    training_args = TrainingArguments(
        output_dir="../models/member3_distilbert_results",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_dir="../models/member3_distilbert_logs",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none"
    )

print("Training arguments ready")


In [ ]:
# ============================================================
# 21. CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer ready")


In [ ]:
# ============================================================
# 22. TRAIN DISTILBERT MODEL
# ============================================================

trainer.train()


In [ ]:
# ============================================================
# 23. EVALUATE DISTILBERT MODEL
# ============================================================

bert_results = trainer.evaluate()

print("DistilBERT Evaluation Results")
print("-----------------------------")
print(bert_results)


In [ ]:
# ============================================================
# 24. DISTILBERT CLASSIFICATION REPORT
# ============================================================

predictions_output = trainer.predict(test_dataset)

logits = predictions_output.predictions
y_pred_bert = np.argmax(logits, axis=1)
y_true_bert = predictions_output.label_ids

print("DistilBERT Classification Report")
print("--------------------------------")
print(classification_report(
    y_true_bert,
    y_pred_bert,
    target_names=bert_label_encoder.classes_,
    zero_division=0
))


In [ ]:
# ============================================================
# 25. DISTILBERT CONFUSION MATRIX
# ============================================================

cm_bert = confusion_matrix(y_true_bert, y_pred_bert)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm_bert,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=bert_label_encoder.classes_,
    yticklabels=bert_label_encoder.classes_
)

plt.title("DistilBERT Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 26. SAVE DISTILBERT MODEL
# ============================================================

distilbert_model.save_pretrained("../models/member3_distilbert_model")
tokenizer.save_pretrained("../models/member3_distilbert_model")

joblib.dump(bert_label_encoder, "../models/member3_bert_label_encoder.pkl")

print("Member 3 DistilBERT model saved successfully")


In [ ]:
# ============================================================
# 27. DISTILBERT SINGLE PREDICTION FUNCTION
# ============================================================

def predict_with_distilbert(title, abstract):
    text = title + " " + abstract

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    distilbert_model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    distilbert_model.eval()

    with torch.no_grad():
        outputs = distilbert_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    prediction = np.argmax(probabilities)
    predicted_category = bert_label_encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction] * 100

    return predicted_category, confidence

category, confidence = predict_with_distilbert(sample_title, sample_abstract)

print("DistilBERT Prediction")
print("---------------------")
print("Predicted Category:", category)
print("Confidence:", round(confidence, 2), "%")


In [ ]:
# ============================================================
# 28. DISTILBERT TOP-3 PREDICTION FUNCTION
# ============================================================

def predict_top3_with_distilbert(title, abstract):
    text = title + " " + abstract

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    distilbert_model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    distilbert_model.eval()

    with torch.no_grad():
        outputs = distilbert_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    top3_indices = np.argsort(probabilities)[-3:][::-1]

    results = []

    for idx in top3_indices:
        category = bert_label_encoder.inverse_transform([idx])[0]
        confidence = probabilities[idx] * 100
        results.append((category, round(confidence, 2)))

    return results

top3_bert = predict_top3_with_distilbert(sample_title, sample_abstract)

print("DistilBERT Top-3 Predictions")
print("----------------------------")
for category, confidence in top3_bert:
    print(category, ":", confidence, "%")


In [ ]:
# ============================================================
# 29. FINAL MEMBER 3 SUMMARY
# ============================================================

print("Member 3 Model Development Completed")
print("------------------------------------")
print("ML Model: XGBoost with TF-IDF")
print("DL Model: DistilBERT")
print("Dataset: arxiv_15000_balanced.csv")
print("Classes:", list(bert_label_encoder.classes_))
